# Career Compass — Data Exploration
Initial look at the LinkedIn Job Postings dataset for the job recommendation platform.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

BASE = "./"  # postings.csv and subfolders live alongside this notebook

## Postings

Core table: one row per job posting, with `title`, `description` (free text — this is the NLP source), salary fields, location, experience level, and engagement stats (`views`, `applies`).

In [ ]:
postings = pd.read_csv(f"{BASE}postings.csv")
print("shape:", postings.shape)
postings.head()

In [ ]:
missing_pct = (postings.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct

Notable gaps: `closed_time` (99.1%), `skills_desc` (98.0%), `med_salary` (94.9%), `remote_allowed` (87.7%), and the salary trio `min_salary`/`max_salary`/`normalized_salary` (~76%) are mostly missing. Salary and remote-work analyses will need to handle this sparsity explicitly rather than dropping rows.

## Skills & industries mappings

`jobs/job_skills.csv` joins to postings via `job_id`, but `mappings/skills.csv` only has **35 broad functional categories** (e.g. "Information Technology", "Sales", "Engineering") — not granular skills like "Python" or "SQL". Real skill extraction for the recommendation engine has to come from parsing the `description` text itself.

In [ ]:
job_skills = pd.read_csv(f"{BASE}jobs/job_skills.csv")
skills_map = pd.read_csv(f"{BASE}mappings/skills.csv")
industries_map = pd.read_csv(f"{BASE}mappings/industries.csv")
job_industries = pd.read_csv(f"{BASE}jobs/job_industries.csv")

skills_joined = job_skills.merge(skills_map, on="skill_abr", how="left")
print("job_skills shape:", job_skills.shape)
print("coverage of postings:", postings.job_id.isin(job_skills.job_id).mean().round(3))

skills_joined["skill_name"].value_counts().head(20)

## Companies

Company metadata (size, location, description) can enrich recommendations beyond raw job text.

In [ ]:
companies = pd.read_csv(f"{BASE}companies/companies.csv")
print("shape:", companies.shape)
companies.head()

## Next steps

- Build a skill gazetteer/dictionary (or NER model) to extract granular skills from `description`.
- Decide how to handle salary sparsity (~76% missing) — likely restrict salary-based analyses to the subset with data, and flag this limitation in the report.
- Explore `formatted_experience_level` and `location` distributions to understand the postings population.
- Prototype the recommendation similarity approach (e.g. TF-IDF/embeddings over extracted skills) on a small sample before scaling to the full 123k postings.